# Verifiable Quantum Circuit Compilation

## v9 · The 27.30 Frontier: Fresher Weights and Our Scanners on the New Circuits

Notebook 01 v3 submits the public frontier as published (≈ 27.30). This notebook starts from exactly that portfolio and
tests three cheap hypotheses, each checked by the **unmodified official verifiers** before it is used:

| # | Hypothesis | Where | Expected size |
|---|---|---|---|
| H1 | A newer public W/H2_I continuation (Atakan Aldemir reports −503 cost units vs Heliosli V52) fits the same words and loaders | chemistry `submission.npz` | tiny (+0.00002) |
| H2 | Our notebook-08 scanners find Toffolis in the **new 46-qubit invsqrt**, which no one has run them on: GF(2) quadratic-form merges (Dickson rank/2, shared products), dirty → clean three-input ANDs, and dead writes (asteriospanoras' motif, every CCX) | `invsqrt.kmb` | ≈ +0.006 per Toffoli |
| H3 | The same scanners on maoxin123's fifteen 4,038-Toffoli loaders | `angle_k_*.kmb` | ≈ +0.001 per Toffoli (each counts twice) |

Each rewrite is exact on all inputs (no reachable-state assumption), adds no qubits, and is kept only if the verifier
passes it with a lower cost. The notebook always writes a `submission.zip`: the best verified combination, or the
unchanged frontier. DQI is not scanned: its 53M-operation text does not fit in a CPU session (notebook 08).

**Kaggle setup:** Accelerator *None* (CPU only), Internet off. Inputs: the competition data and the outputs of Atakan
Aldemir's, Heliosli's and Areeb Ali's notebooks (as in notebook 01 v3). Runtime ≈ 40–60 minutes.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────────────────────
SALT = 'flexonafft-v9'
PINS = {'invsqrt.kmb': 'dcde7078', 'dqi.kmb': '821bf8d1', 'dfthcsos/submission.npz': '2f0cb894',
        'dfthcsos/angle_k_0.kmb': '93fffb08'}
REFERENCE_COST = {'invsqrt': 2_076_774.233715282, 'dqi': 24_439_463_583_672.93, 'dfthcsos': 3_355_599_588.3155365}
TASKS = ('invsqrt', 'dqi', 'dfthcsos')
SCAN = dict(rounds=8, maxrun=10, maxspan=6000, pair_window=400)
WORKERS = 4

In [ ]:
# ── Session ───────────────────────────────────────────────────────────────────────────────────
import os, sys, glob, json, time, math, shutil, hashlib, zipfile, itertools
from pathlib import Path
import pandas as pd

NB_T0 = time.time()
ROOT = Path('/kaggle/working'); ROOT.mkdir(parents=True, exist_ok=True); os.chdir(ROOT)
INPUT = Path('/kaggle/input')
def deep_find(pattern):
    return sorted(Path(p) for p in glob.glob(str(INPUT / '**' / pattern), recursive=True))
KICK = deep_find('kickmix*.so')[0].parent
VERIF = deep_find('verify_submission_invsqrt.py')[0].parent.parent
sys.path[:0] = [str(KICK), str(VERIF)]
import kickmix as km
from invsqrt import verify_submission_invsqrt as v_invsqrt
from dqi import verify_submission_dqi as v_dqi
from dfthcsos import verify as v_dfthcsos
sha = lambda b: hashlib.sha256(b).hexdigest()
def log(*a): print(f'[{(time.time() - NB_T0) / 60:5.1f} min]', *a, flush=True)

## The base: notebook 01 v3's portfolio

Same assembly as notebook 01 v3: files indexed by SHA-256 across every attached `submission.zip`, the loader family
taken whole from one zip, then asteriospanoras' dead-write fix in invsqrt.

In [ ]:
POOL = {}
for z in deep_find('submission.zip'):
    with zipfile.ZipFile(z) as Z:
        for n in Z.namelist():
            if not n.endswith('/'):
                b = Z.read(n)
                POOL.setdefault(n, {}).setdefault(sha(b), (z, b))
def pinned(name):
    hits = [(h, v) for h, v in POOL.get(name, {}).items() if h.startswith(PINS[name])]
    assert hits, f'{name} {PINS[name]} not found; available: {[h[:8] for h in POOL.get(name, {})]}'
    return hits[0]
BASE = ROOT / 'base'; shutil.rmtree(BASE, ignore_errors=True); (BASE / 'dfthcsos').mkdir(parents=True)
for name in PINS:
    (BASE / name).write_bytes(pinned(name)[1][1])
loader_zip = pinned('dfthcsos/angle_k_0.kmb')[1][0]
with zipfile.ZipFile(loader_zip) as Z:
    for k in range(1, 15):
        (BASE / f'dfthcsos/angle_k_{k}.kmb').write_bytes(Z.read(f'dfthcsos/angle_k_{k}.kmb'))

def to_lines(path):
    km.Circuit.from_file(str(path), format='kmb').to_file('/tmp/x.kmx', format='kmx')
    return open('/tmp/x.kmx').read().splitlines()
def from_lines(lines, path):
    open('/tmp/y.kmx', 'w').write('\n'.join(lines) + '\n')
    km.Circuit.from_file('/tmp/y.kmx', format='kmx').to_file(str(path), format='kmb')

lines = to_lines(BASE / 'invsqrt.kmb')
ol = [i for i, l in enumerate(lines) if l.split() and l.split()[0] not in ('APPEND_TO_REGISTER', 'REGISTER', 'INFO_TAG', 'DEBUG_PRINT')]
i1, i2 = ol[3292], ol[3403]
assert lines[i1] == 'CCX q16 q20 q21' and lines[i2] == 'HMR q21 b61'
lines = lines[:i1] + ['HMR q21 b61', 'CZ q16 q20 if b61'] + lines[i1 + 1:i2] + lines[i2 + 1:]
from_lines(lines, BASE / 'invsqrt.kmb')
FILES = sorted(p.relative_to(BASE).as_posix() for p in BASE.rglob('*') if p.is_file())
assert len(FILES) == 18
log('base portfolio', {f: sha((BASE / f).read_bytes())[:8] for f in ('invsqrt.kmb', 'dqi.kmb', 'dfthcsos/submission.npz')})

## H1 · Chemistry weights

Every distinct `dfthcsos/submission.npz` among the attached outputs is tried with the pinned loaders. The verifier
decodes the angle words from the circuits, so a table whose words do not match the loaders simply fails.

In [ ]:
def chem_verify(npz_bytes, loader_dir):
    d = ROOT / 'chem_try'; shutil.rmtree(d, ignore_errors=True); shutil.copytree(loader_dir, d)
    (d / 'submission.npz').write_bytes(npz_bytes)
    try:
        return v_dfthcsos.verify(str(d), salt=SALT)
    except Exception as ex:
        return {'status': 'ERROR', 'error': repr(ex)[:300]}

CHEM = []
for h, (z, b) in POOL['dfthcsos/submission.npz'].items():
    r = chem_verify(b, BASE / 'dfthcsos')
    CHEM.append(dict(sha=h[:12], source=str(z.relative_to(INPUT)), status=r['status'], cost=float(r.get('score', 'nan')),
                     lambda_eff=r.get('lambda_eff'), T=r.get('toffoli_count'), Q=r.get('num_qubits'), npz=b))
    log('npz', h[:12], r['status'], r.get('score'))
chem_df = pd.DataFrame([{k: v for k, v in c.items() if k != 'npz'} for c in CHEM]).sort_values('cost')
display(chem_df)
ok = [c for c in CHEM if c['status'] == 'PASS']
BEST_NPZ = min(ok, key=lambda c: c['cost'])
BASE_CHEM = next(c for c in CHEM if c['sha'].startswith(PINS['dfthcsos/submission.npz']))
print(f"H1: best npz {BEST_NPZ['sha']} cost {BEST_NPZ['cost']:.9g} vs pinned {BASE_CHEM['cost']:.9g} "
      f"({BEST_NPZ['cost'] / BASE_CHEM['cost'] - 1:+.3e})")

## The scanners

`qmerge.py` is notebook 08's scanner plus a generalised dead-write detector. Its self-test re-checks random wasteful
circuits on every input before and after rewriting, and Ehan Mallick's 3 → 2 window.

In [ ]:
QMERGE_SOURCE = r'''"""qmerge: find XOR-accumulated AND products that can be re-synthesised with fewer Toffolis.

A qubit T that, over a stretch of the circuit, only receives CCX/CX/X targets and is not read, collects a GF(2)
polynomial of degree <= 2 in the values of the other wires. Dickson's theorem: such a quadratic form needs exactly
rank/2 AND gates. Two such targets can also share products: f1 = f2 + (f1 + f2), with the f2 products fanned out into
both targets by CX conjugation. This generalises Ehan Mallick's two hand-found 3->2 merges on invsqrt (qf2).

Semantics used (kickmix ops): X, CX, SWAP, CCX are classical permutations; Z, CZ, CCZ, Z_POW are diagonal (do not
change basis values); HMR and R discard a qubit; everything under PUSH_CONDITION, and every other op that touches a
qubit, is treated as an unknown change of that qubit. A product term may be moved in time past any op that does not
touch its target, except an HMR/R on a qubit its value depends on; this is checked by a span test at every such op.
"""
import re
from collections import defaultdict

LAST = {}
QRE = re.compile(r'\bq(\d+)\b')
DIAG = {'Z', 'CZ', 'CCZ', 'Z_POW'}
PASS = {'INFO_TAG', 'DEBUG_PRINT', 'BIT_INVERT', 'BIT_STORE0', 'BIT_STORE1'}
DISCARD = {'HMR', 'R'}


# ── GF(2) polynomials of degree <= 2: (quad: set of (i<j), lin: set, const: int); affine form: (frozenset, int) ──
def mul(A, B):
    (a, a0), (b, b0) = A, B
    quad, lin = set(), set()
    for x in a:
        for y in b:
            if x == y:
                lin ^= {x}
            else:
                quad ^= {(x, y) if x < y else (y, x)}
    if a0:
        lin ^= set(b)
    if b0:
        lin ^= set(a)
    return quad, lin, a0 & b0


def add_into(P, Q):
    P[0].symmetric_difference_update(Q[0]); P[1].symmetric_difference_update(Q[1]); P[2] ^= Q[2]


def poly_of(entries):
    P = [set(), set(), 0]
    for e in entries:
        add_into(P, mul(e[1], e[2]))
    return P


def dickson(P):
    """f = sum_i L_i M_i + affine, with rank/2 products (minimal for one quadratic form)."""
    quad, lin, c = set(P[0]), set(P[1]), P[2]
    prods = []
    while quad:
        i, j = min(quad)
        A, B = set(), set()
        for (u, v) in [t for t in quad if i in t or j in t]:
            quad.discard((u, v))
            if (u, v) == (i, j):
                continue
            if i in (u, v):
                A ^= {v if u == i else u}
            else:
                B ^= {v if u == j else u}
        a0 = 1 if i in lin else 0
        b0 = 1 if j in lin else 0
        lin.discard(i); lin.discard(j)
        prods.append(((frozenset({i}) ^ frozenset(B), b0), (frozenset({j}) ^ frozenset(A), a0)))
        q2, l2, c2 = mul((frozenset(A), a0), (frozenset(B), b0))
        quad ^= q2; lin ^= l2; c ^= c2
    return prods, (frozenset(lin), c)


def evaluate(P, x):
    quad, lin, c = P
    s = c
    for (u, v) in quad:
        s ^= x[u] & x[v]
    for u in lin:
        s ^= x[u]
    return s


# ── Parsing ──
def parse(lines):
    ops = []
    for i, raw in enumerate(lines):
        s = raw.split('#', 1)[0].strip()
        if not s:
            continue
        tok = s.split(None, 1)
        op = tok[0].split('(', 1)[0]
        qs = [int(m) for m in QRE.findall(tok[1])] if len(tok) > 1 else []
        if ' if ' in f' {s} ' and op not in ('PUSH_CONDITION', 'POP_CONDITION'):
            op = 'IF_' + op          # classically conditioned: handled as an unknown change of its qubits
        ops.append((i, op, qs))
    return ops


def stats(ops):
    h = defaultdict(int)
    for _, op, _ in ops:
        h[op] += 1
    return dict(h)


# ── Symbolic pass ──
class Tracker:
    def __init__(self):
        self.form = {}
        self.birth = []
        self.acc = {}

    def fresh(self, q, line):
        v = len(self.birth)
        self.birth.append(line)
        self.form[q] = (frozenset((v,)), 0)

    def get(self, q):
        if q not in self.form:
            self.fresh(q, -1)
        return self.form[q]


def run(ops, want=()):
    """One pass. Returns intervals (T, entries, start, end), discard events and snapshots at the lines in `want`."""
    want = set(want)
    t = Tracker()
    form, acc = t.form, t.acc
    intervals, discards, snaps = [], [], {}
    depth = 0
    for q in sorted({q for _, _, qs in ops for q in qs}):
        t.fresh(q, -1)

    def close(q, line):
        a = acc.pop(q, None)
        if a is not None:
            if a['entries']:
                intervals.append((q, a['entries'], a['start'], line))
            t.fresh(q, line - 0.5)

    for (i, op, qs) in ops:
        if i in want:
            snaps[i] = ({q: f for q, f in form.items() if q not in acc}, frozenset(acc))
        if op == 'PUSH_CONDITION':
            depth += 1
            for q in qs:
                close(q, i)
            continue
        if op == 'POP_CONDITION':
            depth -= 1
            continue
        if op in PASS or not qs:
            continue
        for q in qs:
            t.get(q)
        if depth > 0:
            for q in qs:
                close(q, i)
                if op not in DIAG:
                    t.fresh(q, i)
            continue
        if op.startswith('IF_'):
            for q in qs:
                close(q, i)
                if op[3:] not in DIAG:
                    t.fresh(q, i)
            continue
        if op == 'X':
            for q in qs:
                if q not in acc:
                    fs, c = form[q]; form[q] = (fs, c ^ 1)
        elif op == 'CX' and len(qs) % 2 == 0:
            for k in range(0, len(qs), 2):
                c_, tg = qs[k], qs[k + 1]
                close(c_, i)
                if tg not in acc:
                    a, b = form[c_], form[tg]
                    form[tg] = (a[0] ^ b[0], a[1] ^ b[1])
        elif op == 'SWAP' and len(qs) % 2 == 0:
            for k in range(0, len(qs), 2):
                a, b = qs[k], qs[k + 1]
                close(a, i); close(b, i)
                form[a], form[b] = form[b], form[a]
        elif op == 'CCX' and len(qs) == 3:
            a, b, tg = qs
            close(a, i); close(b, i)
            if tg not in acc:
                acc[tg] = dict(start=i, entries=[])
            acc[tg]['entries'].append((i, form[a], form[b], tg))
        elif op in DIAG:
            for q in qs:
                close(q, i)
        elif op in DISCARD:
            for q in qs:
                close(q, i)
                discards.append((i, q))
                form[q] = (frozenset(), 0)      # R / HMR leave the wire in |0>
        else:
            for q in qs:
                close(q, i)
                t.fresh(q, i)
    end = ops[-1][0] + 1 if ops else 0
    for q in list(acc):
        close(q, end)
    return intervals, discards, snaps, t.birth


# ── Linear algebra on snapshots ──
def echelon(vecs):
    varidx, piv = {}, {}
    for idx, (q, fs) in enumerate(vecs):
        m = 0
        for v in fs:
            m |= 1 << varidx.setdefault(v, len(varidx))
        c = 1 << idx
        while m:
            hb = m.bit_length() - 1
            if hb in piv:
                pm, pc = piv[hb]; m ^= pm; c ^= pc
            else:
                piv[hb] = (m, c); break
    return varidx, piv, vecs


def represent(ech, fs):
    varidx, piv, vecs = ech
    m = 0
    for v in fs:
        if v not in varidx:
            return None
        m |= 1 << varidx[v]
    c = 0
    while m:
        hb = m.bit_length() - 1
        if hb not in piv:
            return None
        pm, pc = piv[hb]; m ^= pm; c ^= pc
    return [vecs[k][0] for k in range(len(vecs)) if c >> k & 1]


def basis(snap, excl):
    forms, accset = snap
    return {q: f for q, f in forms.items() if q not in excl and q not in accset}


def make_ech(forms):
    return echelon([(q, f[0]) for q, f in sorted(forms.items()) if f[0]])


# ── Candidate groups ──
def nprod(P):
    return len(dickson(P)[0])


def candidates(intervals, maxrun=10, maxspan=4000, pair_window=300, maxpairs=200000):
    """Return groups: dict(targets, entries per target, plan, saving). plan: list of (poly, target tuple)."""
    out = []
    by_entry = []
    for (T, ents, s, e) in intervals:
        n = len(ents)
        for a in range(n):
            P = [set(), set(), 0]
            for b in range(a, min(n, a + maxrun)):
                if ents[b][0] - ents[a][0] > maxspan:
                    break
                add_into(P, mul(ents[b][1], ents[b][2]))
                k = b - a + 1
                if k >= 2:
                    h = nprod(P)
                    if h < k:
                        out.append(dict(targets=(T,), runs={T: ents[a:b + 1]}, iv={T: (s, e)},
                                        plan=[([set(P[0]), set(P[1]), P[2]], (T,))], saving=k - h))
        for en in ents:
            by_entry.append((en[0], T, s, e, en))
    # pairs: entries of two targets within pair_window lines sharing a variable
    by_entry.sort(key=lambda r: r[0])
    seen = set()
    pairs = 0
    lines = [r[0] for r in by_entry]
    import bisect
    ivmap = {}
    for (T, ents, s, e) in intervals:
        ivmap[(T, s)] = (ents, s, e)
    for idx, (l1, T1, s1, e1, en1) in enumerate(by_entry):
        v1 = en1[1][0] | en1[2][0]
        hi = bisect.bisect_right(lines, l1 + pair_window)
        for jdx in range(idx + 1, hi):
            l2, T2, s2, e2, en2 = by_entry[jdx]
            if T2 == T1 or (T1, s1, T2, s2) in seen or (T2, s2, T1, s1) in seen:
                continue
            if not (v1 & (en2[1][0] | en2[2][0])):
                continue
            if max(s1, s2) >= min(e1, e2):
                continue
            seen.add((T1, s1, T2, s2))
            pairs += 1
            if pairs > maxpairs:
                break
            lo, up = min(l1, l2) - pair_window, max(l1, l2) + pair_window
            r1 = [x for x in ivmap[(T1, s1)][0] if lo <= x[0] <= up][:maxrun]
            r2 = [x for x in ivmap[(T2, s2)][0] if lo <= x[0] <= up][:maxrun]
            f1, f2 = poly_of(r1), poly_of(r2)
            f12 = [f1[0] ^ f2[0], f1[1] ^ f2[1], f1[2] ^ f2[2]]
            h1, h2, h12 = nprod(f1), nprod(f2), nprod(f12)
            k = len(r1) + len(r2)
            opts = [(h1 + h2, [(f1, (T1,)), (f2, (T2,))]),
                    (h2 + h12, [(f2, (T1, T2)), (f12, (T1,))]),
                    (h1 + h12, [(f1, (T1, T2)), (f12, (T2,))])]
            best = min(opts, key=lambda o: o[0])
            if best[0] < k:
                out.append(dict(targets=(T1, T2), runs={T1: r1, T2: r2}, iv={T1: (s1, e1), T2: (s2, e2)},
                                plan=best[1], saving=k - best[0]))
        if pairs > maxpairs:
            break
    return out


# ── Validation and emission ──
def group_lines(g):
    return sorted(x[0] for r in g['runs'].values() for x in r)


def needed_lines(g, discards_sorted):
    import bisect
    ls = group_lines(g)
    lo, hi = ls[0], ls[-1]
    dl = [d[0] for d in discards_sorted]
    a, b = bisect.bisect_left(dl, lo), bisect.bisect_right(dl, hi)
    return set(ls) | {discards_sorted[k][0] for k in range(a, b)}


def emit_term(snapforms, poly, targets):
    """Return list of kmx lines that XOR poly into each target, or None if not constructible here."""
    prods, rem = dickson(poly)
    T0 = targets[0]
    out = []
    pre = [f'CX q{T0} q{T}' for T in targets[1:]]
    for (L, M) in prods:
        forms = dict(snapforms)
        best = None
        ech = make_ech(forms)
        RL = represent(ech, L[0])
        if not RL:
            return None
        for a in sorted(RL, key=lambda q: len(forms[q][0])):
            opsA = [f'CX q{q} q{a}' for q in RL if q != a]
            cL = 0
            for q in RL:
                cL ^= forms[q][1]
            if cL != L[1]:
                opsA.append(f'X q{a}')
            f2 = dict(forms); f2[a] = L
            RM = represent(make_ech(f2), M[0])
            if not RM:
                continue
            cands = [b for b in RM if b != a]
            if not cands:
                continue
            b = min(cands, key=lambda q: len(f2[q][0]))
            opsB = [f'CX q{q} q{b}' for q in RM if q != b]
            cM = 0
            for q in RM:
                cM ^= f2[q][1]
            if cM != M[1]:
                opsB.append(f'X q{b}')
            best = opsA + opsB + [f'CCX q{a} q{b} q{T0}'] + opsB[::-1] + opsA[::-1]
            break
        if best is None:
            return None
        out += best
    if rem[0] or rem[1]:
        R = represent(make_ech(snapforms), rem[0]) if rem[0] else []
        if R is None:
            return None
        out += [f'CX q{q} q{T0}' for q in R]
        c = rem[1]
        for q in R:
            c ^= snapforms[q][1]
        if c:
            out.append(f'X q{T0}')
    return pre + out + pre[::-1]


def term_factors(poly):
    """Affine forms whose span carries the polynomial's value (Dickson factors and remainder)."""
    prods, rem = dickson(poly)
    fs = [f for pm in prods for f in pm]
    if rem[0]:
        fs.append(rem)
    return fs


def validate(g, snaps, discards_sorted, birth):
    """Pick a reference time tau and placements; return dict(line -> block lines, removed lines) or None."""
    import bisect
    targets = g['targets']

    def excl_at(line):
        return {T for T in targets if g['iv'][T][0] <= line}
    ls = group_lines(g)
    dl = [d[0] for d in discards_sorted]
    entries = [x for r in g['runs'].values() for x in r]

    def born_ok(forms, line):
        return all(birth[v] < line for f in forms for v in f[0])

    def movable(forms, p, tau):
        if not born_ok(forms, min(p, tau)):
            return False
        lo, hi = min(p, tau), max(p, tau)
        for k in range(bisect.bisect_left(dl, lo), bisect.bisect_left(dl, hi)):
            h, m = discards_sorted[k]
            ech = make_ech(basis(snaps[h], excl_at(h) | {m}))
            for f in forms:
                if f[0] and represent(ech, f[0]) is None:
                    return False
        return True

    def inside(p, ts):
        return all(g['iv'][T][0] <= p < g['iv'][T][1] for T in ts)

    why = defaultdict(int)
    for tau in sorted(ls, key=lambda l: abs(l - ls[len(ls) // 2])):
        if not all(movable([e[1], e[2]], e[0], tau) for e in entries):
            why['entries not movable to tau'] += 1
            continue
        blocks = defaultdict(list)
        ok = True
        for poly, ts in g['plan']:
            if not poly[0] and not poly[1] and not poly[2]:
                continue
            fac = term_factors(poly)
            placed = False
            for p in sorted(ls, key=lambda l: abs(l - tau)):
                if not inside(p, ts) or not movable(fac, p, tau):
                    continue
                code = emit_term(basis(snaps[p], excl_at(p)), poly, ts)
                if code is not None:
                    blocks[p] += code
                    placed = True
                    break
            if not placed:
                why['term not placeable'] += 1
                ok = False
                break
        if ok:
            return dict(blocks=dict(blocks), removed=set(ls), span=(ls[0], ls[-1]))
    g['why'] = dict(why)
    return None


def apply(lines, accepted):
    removed = set().union(*[a['removed'] for a in accepted]) if accepted else set()
    blocks = defaultdict(list)
    for a in accepted:
        for p, b in a['blocks'].items():
            blocks[p] += b
    out = []
    for i, l in enumerate(lines):
        if i in blocks:
            out += blocks[i]
        if i not in removed:
            out.append(l)
    return out


def count_ccx(lines):
    return sum(1 for l in lines if l.split('#', 1)[0].strip().split(None, 1)[:1] == ['CCX'])


def optimise_round(lines, log=print, time_left=lambda: 1e9, maxcands=5000, report=0, **kw):
    ops = parse(lines)
    intervals, discards, _, birth = run(ops)
    cands = candidates(intervals, **kw)
    cands.sort(key=lambda g: (-g['saving'], group_lines(g)[-1] - group_lines(g)[0]))
    log(f'  intervals {len(intervals)}  candidate groups {len(cands)}  '
        f'potential {sum(g["saving"] for g in cands)}')
    if not cands:
        return lines, []
    discards.sort()
    want = set()
    for g in cands[:maxcands]:
        want |= needed_lines(g, discards)
    _, _, snaps, birth2 = run(ops, want=want)
    assert birth2 == birth
    accepted, spans, rejected = [], [], []
    for g in cands[:maxcands]:
        if time_left() < 0:
            break
        lo, hi = group_lines(g)[0], group_lines(g)[-1]
        if any(not (hi < a or lo > b) for a, b in spans):
            continue
        v = validate(g, snaps, discards, birth)
        if v is not None:
            v['saving'] = g['saving']; v['targets'] = g['targets']
            accepted.append(v); spans.append((lo, hi))
        else:
            rejected.append(g)
    chosen = cands[:maxcands]
    LAST['rejected'] = rejected
    for g in rejected[:report]:
        log(f'    rejected targets {g["targets"]} lines {group_lines(g)} saving {g["saving"]} why {g.get("why")}')
    log(f'  shortlisted {len(chosen)}  accepted {len(accepted)}  saving {sum(a["saving"] for a in accepted)}')
    if not accepted:
        return lines, []
    new = apply(lines, accepted)
    assert count_ccx(lines) - count_ccx(new) == sum(a['saving'] for a in accepted), 'CCX accounting'
    return new, [dict(saving=a['saving'], targets=list(a['targets']), span=list(a['span'])) for a in accepted]


def optimise(lines, rounds=8, log=print, time_left=lambda: 1e9, report=0, **kw):
    history = []
    for r in range(rounds):
        log(f'round {r}')
        lines, acc = optimise_round(lines, log=log, time_left=time_left, report=report if r == 0 else 0, **kw)
        history += acc
        if not acc or time_left() < 0:
            break
    return lines, history


# ── Self-test: classical simulation of X/CX/CCX/SWAP circuits ──
def simulate(lines, n, x):
    s = list(x)
    for _, op, qs in parse(lines):
        if op == 'X':
            for q in qs: s[q] ^= 1
        elif op == 'CX':
            for k in range(0, len(qs), 2): s[qs[k + 1]] ^= s[qs[k]]
        elif op == 'CCX':
            s[qs[2]] ^= s[qs[0]] & s[qs[1]]
        elif op == 'SWAP':
            s[qs[0]], s[qs[1]] = s[qs[1]], s[qs[0]]
    return s


def self_test(seed=1, trials=300):
    import random, itertools
    rnd = random.Random(seed)
    # Dickson identity
    for _ in range(trials):
        n = rnd.randint(2, 9)
        P = [set(), set(), rnd.randint(0, 1)]
        for _ in range(rnd.randint(1, 12)):
            u, v = rnd.sample(range(n), 2); P[0] ^= {(min(u, v), max(u, v))}
        P[1] = {u for u in range(n) if rnd.random() < .4}
        prods, rem = dickson(P)
        for _ in range(16):
            x = [rnd.randint(0, 1) for _ in range(n)]
            s = rem[1] ^ (sum(x[v] for v in rem[0]) & 1)
            for L, M in prods:
                s ^= ((sum(x[v] for v in L[0]) + L[1]) & 1) & ((sum(x[v] for v in M[0]) + M[1]) & 1)
            assert s == evaluate(P, x)
    # Random wasteful circuits: equivalence on every input
    found = 0
    for tr in range(trials):
        n = rnd.randint(5, 8)
        lines = []
        for _ in range(rnd.randint(8, 30)):
            r = rnd.random()
            if r < .45:
                a, b = rnd.sample(range(n), 2); lines.append(f'CX q{a} q{b}')
            elif r < .9:
                a, b, c = rnd.sample(range(n), 3); lines.append(f'CCX q{a} q{b} q{c}')
            elif r < .95:
                lines.append(f'X q{rnd.randrange(n)}')
            else:
                a, b = rnd.sample(range(n), 2); lines.append(f'SWAP q{a} q{b}')
        new, hist = optimise(lines, rounds=4, log=lambda *a: None, maxspan=100, pair_window=30)
        found += sum(h['saving'] for h in hist)
        assert count_ccx(lines) - count_ccx(new) == sum(h['saving'] for h in hist)
        for x in itertools.product((0, 1), repeat=n):
            assert simulate(lines, n, x) == simulate(new, n, x), (lines, new)
    return found


# ── Dirty-ancilla three-input AND -> clean ancilla ──
# Pattern (4 CCX): CX d y; CCX a b d; CCX d y T; CX d y; CCX a b d; CCX d y T; CX d T
# Net effect: T ^= a*b*y0, y ^= a*b, d unchanged. With a clean |0> wire z it is 2 CCX + measurement uncompute,
# or 3 CCX with a plain uncompute.
def _toks(l):
    s = l.split('#', 1)[0].strip().split()
    return s


def c3x_sites(lines):
    sites = []
    T = [_toks(l) for l in lines]
    for s in range(len(lines) - 6):
        w = T[s:s + 7]
        try:
            if [x[0] for x in w] != ['CX', 'CCX', 'CCX', 'CX', 'CCX', 'CCX', 'CX'] or any(len(x) != (3 if x[0] == 'CX' else 4) for x in w):
                continue
        except IndexError:
            continue
        d, y = w[0][1], w[0][2]
        a, b, d1 = w[1][1], w[1][2], w[1][3]
        if d1 != d or w[2][1:3] not in ([d, y], [y, d]):
            continue
        Tq = w[2][3]
        if w[3][1:] != [d, y] or sorted(w[4][1:3]) != sorted([a, b]) or w[4][3] != d:
            continue
        if sorted(w[5][1:3]) != sorted([d, y]) or w[5][3] != Tq or w[6][1:] != [d, Tq]:
            continue
        if len({a, b, d, y, Tq}) != 5:
            continue
        sites.append(dict(line=s, a=a, b=b, d=d, y=y, T=Tq))
    return sites


def clean_wires(lines, s, e):
    """Qubits whose last touch before line s is an unconditional R/HMR and that are untouched in [s, e]."""
    used = set()
    for l in lines[s:e + 1]:
        used |= set(QRE.findall(l))
    last = {}
    depth = 0
    for l in lines[:s]:
        t = _toks(l)
        if not t:
            continue
        if t[0] == 'PUSH_CONDITION':
            depth += 1
        elif t[0] == 'POP_CONDITION':
            depth -= 1
        for q in QRE.findall(l):
            last[q] = (t[0] in ('R', 'HMR') and depth == 0 and ' if ' not in f' {l} ')
    return sorted((int(q) for q, ok in last.items() if ok and q not in used))


def max_bit(lines):
    m = -1
    for l in lines:
        for x in re.findall(r'\bb(\d+)\b', l):
            m = max(m, int(x))
    return m


def linear_clean(lines, ops, st):
    """A wire z untouched in the window whose value is an XOR of wires that are unchanged across it:
    returns (z, cleaning ops) or None. Cleaning: z ^= that XOR (so z = const), undo after the block."""
    s = st['line']
    window = set()
    for l in lines[s:s + 7]:
        window |= {int(q) for q in QRE.findall(l)}
    keep = {int(st[k][1:]) for k in ('a', 'b', 'd')}
    _, _, snaps, _ = run(ops, want={s})
    forms, accset = snaps[s]
    for z in sorted(forms):
        if z in window or z in accset:
            continue
        others = {q: f for q, f in forms.items() if q != z and q not in accset and (q not in window or q in keep)}
        R = represent(make_ech(others), forms[z][0]) if forms[z][0] else []
        if R is None:
            continue
        c = forms[z][1]
        for q in R:
            c ^= others[q][1]
        clean = [f'CX q{q} q{z}' for q in R] + (['X q%d' % z] if c else [])
        return z, clean
    return None


def c3x_rewrite(lines, mbu=True, linear=False):
    sites = c3x_sites(lines)
    ops = parse(lines) if linear else None
    nb = max_bit(lines) + 1
    plan = []
    for st in sites:
        zs = clean_wires(lines, st['line'], st['line'] + 6)
        pre = []
        if zs:
            z = f'q{zs[0]}'
        elif linear:
            lc = linear_clean(lines, ops, st)
            if lc is None:
                st['clean'] = None
                continue
            z, pre = f'q{lc[0]}', lc[1]
        else:
            st['clean'] = None
            continue
        st['clean'] = z; st['cleaning'] = pre
        a, b, y, T = st['a'], st['b'], st['y'], st['T']
        if mbu:
            block = [f'CCX {a} {b} {z}', f'CCX {z} {y} {T}', f'CX {z} {y}', f'HMR {z} b{nb}', f'CZ {a} {b} if b{nb}']
            nb += 1
        else:
            block = [f'CCX {a} {b} {z}', f'CCX {z} {y} {T}', f'CX {z} {y}', f'CCX {a} {b} {z}']
        plan.append((st['line'], pre + block + pre[::-1]))
    out, k = [], 0
    starts = {s: b for s, b in plan}
    i = 0
    while i < len(lines):
        if i in starts:
            out += starts[i]; i += 7
        else:
            out.append(lines[i]); i += 1
    return out, sites


# ── Dead writes (asteriospanoras' motif, generalised to every CCX) ──
# CCX a b t whose target t is touched next only by an unconditional HMR t bK, with a, b, t and bK untouched in between:
# the Toffoli's only lasting effect is the sign (-1)^(m*a*b), so measure first and apply CZ a b if bK (0 Toffolis).
def dead_writes(lines):
    toks = [_toks(l) for l in lines]
    depth_at, depth = [], 0
    for t in toks:
        if t and t[0] == 'PUSH_CONDITION':
            depth += 1
        depth_at.append(depth)
        if t and t[0] == 'POP_CONDITION':
            depth -= 1
    nxt = {}                    # qubit/bit token -> index of next line mentioning it (scan backwards)
    found = []
    mentions = [set(x for x in t[1:] if re.fullmatch(r'[qb]\d+', x)) for t in toks]
    for i in range(len(toks) - 1, -1, -1):
        t = toks[i]
        if t and t[0] == 'CCX' and len(t) == 4 and depth_at[i] == 0 and 'if' not in t:
            a, b, tg = t[1], t[2], t[3]
            j = nxt.get(tg)
            if j is not None:
                u = toks[j]
                if (len(u) == 3 and u[0] == 'HMR' and u[1] == tg and depth_at[j] == 0 and 'if' not in u):
                    bit = u[2]
                    between = set().union(*mentions[i + 1:j]) if j > i + 1 else set()
                    if tg not in between and bit not in between:
                        found.append((i, j, a, b, tg, bit))
        for x in mentions[i]:
            nxt[x] = i
    return sorted(found)


def fix_dead_writes(lines):
    found = dead_writes(lines)
    out = list(lines)
    drop = set()
    for (i, j, a, b, tg, bit) in found:
        out[i] = f'HMR {tg} {bit}\nCZ {a} {b} if {bit}'
        drop.add(j)
    out = [l for k, l in enumerate(out) if k not in drop]
    return '\n'.join(out).split('\n'), found
'''
(ROOT / 'qmerge.py').write_text(QMERGE_SOURCE)
sys.path.insert(0, str(ROOT))
import importlib, qmerge; importlib.reload(qmerge)
print('self-test savings on random circuits:', qmerge.self_test(1, 300))
EHAN = ['CX q1 q4', 'CCX q2 q4 q5', 'CX q1 q4', 'CX q2 q4', 'CX q0 q4', 'CCX q3 q4 q5', 'CX q0 q4', 'CX q3 q4',
        'CX q1 q0', 'CCX q3 q0 q2', 'CX q1 q2', 'CX q1 q0']
new, _ = qmerge.optimise(EHAN, log=lambda *a: None)
assert qmerge.count_ccx(new) == 2 and all(qmerge.simulate(EHAN, 6, x) == qmerge.simulate(new, 6, x)
                                         for x in itertools.product((0, 1), repeat=6))
DW = ['CCX q1 q2 q3', 'CX q1 q4', 'HMR q3 b7']
assert qmerge.fix_dead_writes(DW)[0] == ['HMR q3 b7', 'CZ q1 q2 if b7', 'CX q1 q4']

def scan_circuit(lines, tag):
    rep = {}
    lines, dw = qmerge.fix_dead_writes(lines); rep['dead_writes'] = len(dw)
    lines, hist = qmerge.optimise(lines, log=lambda s: None, **SCAN); rep['merges'] = sum(h['saving'] for h in hist)
    lines, sites = qmerge.c3x_rewrite(lines, mbu=True, linear=True)
    rep['c3x_sites'] = len(sites); rep['c3x_used'] = sum(1 for s in sites if s.get('clean'))
    return lines, rep

## H2 · invsqrt (46 qubits)

In [ ]:
inv_lines = to_lines(BASE / 'invsqrt.kmb')
n0 = qmerge.count_ccx(inv_lines)
new, REP_INV = scan_circuit(inv_lines, 'invsqrt')
REP_INV['ccx'] = (n0, qmerge.count_ccx(new))
log('invsqrt scan', REP_INV)
INV_CAND = None
if qmerge.count_ccx(new) < n0:
    INV_CAND = ROOT / 'invsqrt_scanned.kmb'; from_lines(new, INV_CAND)

## H3 · The fifteen chemistry loaders

In [ ]:
LOADERS = ROOT / 'loaders'; shutil.rmtree(LOADERS, ignore_errors=True); shutil.copytree(BASE / 'dfthcsos', LOADERS)
REP_LOAD, changed = [], 0
for k in range(15):
    p = LOADERS / f'angle_k_{k}.kmb'
    L = to_lines(p); n0 = qmerge.count_ccx(L)
    new, rep = scan_circuit(L, f'k{k}'); rep.update(k=k, ccx=(n0, qmerge.count_ccx(new)))
    REP_LOAD.append(rep)
    if qmerge.count_ccx(new) < n0:
        from_lines(new, p); changed += 1
display(pd.DataFrame(REP_LOAD).set_index('k'))
log('loaders changed:', changed)

## Official verification and the final portfolio

Chemistry: the best H1 table with the H3 loaders if they pass and are cheaper, else with the pinned loaders. invsqrt:
the H2 candidate if it passes and is cheaper. DQI is the pinned file. All three final members are verified here.

In [ ]:
FINAL = ROOT / 'final'; shutil.rmtree(FINAL, ignore_errors=True); shutil.copytree(BASE, FINAL)
CHOICE, RESULTS = {}, {}

# chemistry
(FINAL / 'dfthcsos/submission.npz').write_bytes(BEST_NPZ['npz'])
RESULTS['dfthcsos'] = v_dfthcsos.verify(str(FINAL / 'dfthcsos'), salt=SALT)
CHOICE['dfthcsos'] = f"npz {BEST_NPZ['sha']}, pinned loaders"
if changed:
    r = chem_verify(BEST_NPZ['npz'], LOADERS)
    log('chemistry with scanned loaders:', r['status'], r.get('score'), r.get('toffoli_count'))
    if r['status'] == 'PASS' and float(r['score']) < float(RESULTS['dfthcsos']['score']):
        for k in range(15):
            shutil.copyfile(LOADERS / f'angle_k_{k}.kmb', FINAL / f'dfthcsos/angle_k_{k}.kmb')
        RESULTS['dfthcsos'] = r; CHOICE['dfthcsos'] = f"npz {BEST_NPZ['sha']}, scanned loaders"
log('chemistry', RESULTS['dfthcsos']['status'], RESULTS['dfthcsos']['score'])

# invsqrt
def inv_verify(path):
    return v_invsqrt.verify(km.Circuit.from_file(str(path), format='kmb'), v_invsqrt.ProblemSpecInvSqrt.for_n_p(10),
                            salt=SALT, max_workers=WORKERS)
CHOICE['invsqrt'] = 'base (dcde7078 + dead-write fix)'
if INV_CAND:
    r = inv_verify(INV_CAND)
    log('invsqrt scanned:', r['status'], r.get('score'), r.get('toffoli_count'), r.get('num_qubits'))
    if r['status'] == 'PASS' and r['num_qubits'] <= 46:
        RESULTS['invsqrt'] = r; shutil.copyfile(INV_CAND, FINAL / 'invsqrt.kmb'); CHOICE['invsqrt'] = 'scanned'
if 'invsqrt' not in RESULTS:
    RESULTS['invsqrt'] = inv_verify(FINAL / 'invsqrt.kmb')
log('invsqrt', RESULTS['invsqrt']['status'], RESULTS['invsqrt']['score'])

# dqi
RESULTS['dqi'] = v_dqi.verify(km.Circuit.from_file(str(FINAL / 'dqi.kmb'), format='kmb'),
                              v_dqi.ProblemSpecDQI.benchmark_instance(), salt=SALT, max_workers=WORKERS)
CHOICE['dqi'] = 'base (821bf8d1)'
log('dqi', RESULTS['dqi']['status'], RESULTS['dqi']['score'])
for t in TASKS:
    assert RESULTS[t]['status'] == 'PASS', (t, RESULTS[t])

In [ ]:
ratios = {t: REFERENCE_COST[t] / float(RESULTS[t]['score']) for t in TASKS}
GM = math.prod(ratios.values()) ** (1 / 3)
display(pd.DataFrame([{'task': t, 'choice': CHOICE[t], 'cost': float(RESULTS[t]['score']), 'ratio': ratios[t],
                       'T': RESULTS[t].get('toffoli_count'), 'Q': RESULTS[t].get('num_qubits')} for t in TASKS]).set_index('task'))
files = sorted(p.relative_to(FINAL).as_posix() for p in FINAL.rglob('*') if p.is_file())
assert files == FILES
ZIP = ROOT / 'submission.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for f in files:
        z.write(FINAL / f, f)
(ROOT / 'report09.json').write_text(json.dumps(dict(
    choice=CHOICE, ratios=ratios, expected_GM=GM, chemistry_tables=[{k: v for k, v in c.items() if k != 'npz'} for c in CHEM],
    invsqrt_scan=REP_INV, loader_scan=REP_LOAD, sha256={f: sha((FINAL / f).read_bytes()) for f in files},
    results={t: {k: v for k, v in r.items() if isinstance(v, (int, float, str))} for t, r in RESULTS.items()},
    minutes=round((time.time() - NB_T0) / 60, 1)), indent=1, default=str))
for d in ('base', 'final', 'loaders', 'chem_try'):
    shutil.rmtree(ROOT / d, ignore_errors=True)
for f in ROOT.glob('*.kmb'):
    f.unlink()
print(f'V9 DONE | GM {GM:.6f} | {CHOICE} | {(time.time() - NB_T0) / 60:.1f} min')

# Sources and credits

- **The base portfolio** is notebook 01 v3's, assembled as asteriospanoras published it
  ([QCC 27.30 | Public Frontier + Free CZ Fix in invsq](https://www.kaggle.com/code/asteriospanoras/qcc-27-30-public-frontier-free-cz-fix-in-invsq));
  its full credits (Syed Asad Ali, Ali Haydar Özdağ, Ertuğrul Özer, Areeb Ali, maoxin123, Heliosli, Atakan Aldemir,
  prvsiyan, Leo Tang, Lera (marculera) and the authors her NOTICE credits, Alex Chilton, manwithacat and the DQI lineage)
  are in notebook 01 v3 and in [asteriospanoras' overview](https://www.kaggle.com/code/asteriospanoras/qcc-27-30-what-works-what-fails-and-why).
- **H1:** the alternative W/H2_I tables are the public outputs of
  [Atakan Aldemir](https://www.kaggle.com/code/atakanaldemir/qcc-gap-3-residue-repair-and-c43-native-compiler) (constrained
  spectral/Schur L-BFGS continuation), [Heliosli](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy) (with
  prvsiyan's refit code) and [Areeb Ali](https://www.kaggle.com/code/areebbb/qai-fewer-toffolis-same-qubits-lb-27-11042), used byte for byte.
- **H2/H3:** the dead-write motif is asteriospanoras'; the quadratic-form merge generalises Ehan Mallick's hand-found 3 → 2
  windows ([invsqrt 20.65](https://www.kaggle.com/code/ehanmallick/invsqrt-20-65-two-3-to-2-and-merges-on-y6225));
  L. E. Dickson (1901) for the canonical form; the scanner code and the clean-ancilla rewrite are ours (notebook 08).
- Tasks, kickmix and verifiers: Google Quantum AI / Qualtran.